# Part 2: Join sleepstudy data with participant context data.

### Merge choice:
- Left merge keeps all orignal sleep records.
NA behaviour:
- If an ID exists in sleepstudy but not in participant_context, the new columns will contain NA values

In [1]:
import pandas as pd

In [2]:
# go up one folder to read data as the script lives inside /scripts
sleep = pd.read_csv("../data/sleepstudy.csv")
context = pd.read_csv("../data/participant_context.csv")

In [3]:
# show dataframes before the merge to compare
print("sleep shape:", sleep.shape)
display(sleep.head(5))

print("context shape:", context.shape)
display(context.head(5))

sleep shape: (1000, 15)


,ID,Age,Gender,Bedtime,WakeupTime,SleepDuration,SleepEfficiency,REMSleepPercentage,DeepSleepPercentage,LightSleepPercentage,Awakenings,CaffeineConsumption,AlcoholConsumption,SmokingStatus,ExerciseFrequency
0,1,80,Female,2025-09-30 7:32,49:16.7,6.283241,0.57,15,35,50,0,25.0,1,Yes,1
1,2,24,Male,2025-06-29 20:59,09:10.2,7.155613,0.91,29,68,3,4,50.0,0,No,2
2,3,37,Male,2025-12-24 21:28,31:34.3,6.050627,0.58,15,35,50,3,50.0,0,No,5
3,4,68,Female,2025-02-22 0:25,26:37.0,7.017791,0.88,28,44,28,1,50.0,0,Yes,4
4,5,58,Male,2025-09-02 12:31,17:46.3,8.764793,0.95,28,40,32,4,25.0,4,No,4


context shape: (947, 10)


,ID,City,WorkType,Chronotype,Device,ScreenTimeHours,StressLevel,BMI,CoffeePreference,SleepAidUse
0,1,Langley,Healthcare,Evening,Oura,5.0,7,21.7,Coffee,NaN
1,2,Vancouver,Retail,Morning,Manual Entry,3.6,8,24.5,Tea,NaN
2,3,Richmond,Retail,Evening,Android App,5.5,6,26.6,Coffee,NaN
3,4,Coquitlam,Student,Evening,Fitbit,3.5,6,28.2,NaN,NaN
4,5,Delta,Retail,Morning,Garmin,5.6,9,24.6,NaN,Melatonin


In [4]:
#checking for unique keys
print("Unique IDs in sleep:", sleep["ID"].nunique(), "out of", len(sleep))
print("Unique IDs in context:", context["ID"].nunique(), "out of", len(context))

Unique IDs in sleep: 1000 out of 1000
Unique IDs in context: 947 out of 947


In [5]:
# check for duplicates
sleep_dupes = sleep["ID"].duplicated().sum()
context_dupes = context["ID"].duplicated().sum()
print("Duplicate IDs in sleep:", sleep_dupes)
print("Duplicate IDs in context:", context_dupes)

Duplicate IDs in sleep: 0
Duplicate IDs in context: 0


In [6]:
#merge WITH indicator to show what matched
merged = pd.merge(
    sleep,
    context,
    how="left",
    on="ID",
    validate="1:1",
    indicator=True
)

In [7]:
#What happened in the merge?
print("Merged shape:", merged.shape)
print("merge breakdown (did IDs match?):")
print(merged["_merge"].value_counts())

Merged shape: (1000, 25)
merge breakdown (did IDs match?):
_merge
both          947
left_only      53
right_only      0
Name: count, dtype: int64


In [8]:
#Show NA behaviour
new_cols = [c for c in context.columns if c != "ID"]
print("NA counts in merged (new) columns:\n")
print(merged[new_cols].isna().sum().sort_values(ascending=False))

print("Percentage of sleep records missing context:",
      round((53 / 1000) * 100, 2), "%")

NA counts in merged (new) columns:

SleepAidUse         739
CoffeePreference    128
BMI                  71
ScreenTimeHours      65
City                 64
Device               53
Chronotype           53
WorkType             53
StressLevel          53
dtype: int64
Percentage of sleep records missing context: 5.3 %


In [9]:
#CLean up helper column for cleaner analysis, no room for confusion
merged = merged.drop(columns=["_merge"])

In [10]:
#output the merged file
merged.to_csv("../data/sleepstudy_merged.csv", index=False)

print("\nSaved merged dataset to data/sleepstudy_merged.csv")
print("Merged shape:", merged.shape)


Saved merged dataset to data/sleepstudy_merged.csv
Merged shape: (1000, 24)


### Interpretation of the Results

A left join was used to preserve all 1000 sleep records.

Of these, 947 participant IDs matched successfully with the context dataset, while 53 sleep records did not have corresponding context data.

Because a left join was used, unmatched records retain their sleep data but contain missing values (NA) in the newly added context columns. No duplicate IDs were found in either dataset, confirming the 1:1 merge assumption.